In [0]:
dbutils.widgets.text("src_val","")
src_v = dbutils.widgets.get("src_val")

In [0]:
df_initial = spark.readStream.format("cloudFiles")\
          .option("cloudFiles.format","csv")\
          .option("cloudFiles.schemaLocation",f"/Volumes/dbt_project_2/bronze/bvol/schema/{src_v}")\
          .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
          .load(f"/Volumes/dbt_project_2/inbound/ivol/{src_v}/")

In [0]:
from pyspark.sql.functions import *
df_final = (
            df_initial.withColumn( "ingestion_timestamp", current_timestamp())
                     .withColumn("source_name", col("_metadata.file_name"))
                     .withColumn("source_path", col("_metadata.file_path"))
           )

In [0]:
(
    df_final.writeStream.format("delta")
            .outputMode("append")
            .trigger(availableNow = True)
            .option("checkpointLocation", f"/Volumes/dbt_project_2/bronze/bvol/checkpoint/{src_v}")
            .toTable(f"dbt_project_2.bronze.{src_v}")
            .awaitTermination()
)